In [2]:
%pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

def mapear_covariancia_negativa(diretorio_dados):
    print("=== INÍCIO DO MAPEAMENTO DE COVARIÂNCIA NEGATIVA ===")
    
    # 1. Carregamento da Matriz de Covariância Anualizada
    caminho_cov = os.path.join(diretorio_dados, "matriz_covariancia_anualizada.csv")
    print("1. A carregar a matriz de covariância global...")
    df_cov = pd.read_csv(caminho_cov, index_col=0)
    
    # 2. Criação da Máscara para o Heatmap
    print("2. A aplicar filtros matriciais (isolando covariâncias negativas)...")
    # Criamos uma máscara que é VERDADEIRA onde a covariância é MAIOR ou IGUAL a zero
    # O Seaborn esconde tudo o que for "Verdadeiro" na máscara.
    mascara_positivos = df_cov >= 0
    
    # Também criamos uma máscara para o triângulo superior (para não duplicar informações no gráfico)
    mascara_triangulo = np.triu(np.ones_like(df_cov, dtype=bool))
    
    # Juntamos as duas máscaras
    mascara_final = mascara_positivos | mascara_triangulo
    
    # 3. Renderização do Heatmap
    print("3. A desenhar o Mapa de Calor (Heatmap)...")
    plt.figure(figsize=(20, 16)) # Tamanho grande devido aos 136 ativos
    
    # Usamos o cmap 'Reds_r' (Vermelhos invertidos) ou 'coolwarm'. 
    # Aqui vou usar 'YlGnBu_r' onde as cores mais escuras mostram a covariância mais negativa
    sns.heatmap(df_cov, 
                mask=mascara_final, 
                cmap="magma",  # Cores vibrantes contra fundo escuro/branco
                center=0, 
                square=True, 
                linewidths=.5, 
                cbar_kws={"shrink": .7, "label": "Nível de Covariância Negativa"})
    
    plt.title('Mapa de Calor: Covariâncias Negativas Isoladas (Proteção Cruzada)', fontsize=16)
    plt.xlabel('Ativos', fontsize=12)
    plt.ylabel('Ativos', fontsize=12)
    
    # Esconde os nomes dos eixos se ficar muito confuso, mas mantemos em tamanho pequeno
    plt.xticks(fontsize=6, rotation=90)
    plt.yticks(fontsize=6)
    
    caminho_grafico = os.path.join(diretorio_dados, "heatmap_covariancia_negativa.png")
    plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Extração Tabela de Pares (O Ouro da Tese)
    print("4. A extrair os pares de ativos com proteção cruzada...")
    
    # Pegamos apenas o triângulo inferior para não ter duplicações (ex: ITUB4-VALE3 e VALE3-ITUB4)
    df_cov_lower = df_cov.where(np.tril(np.ones(df_cov.shape), k=-1).astype(bool))
    
    # Transformamos a matriz numa lista (melt/stack)
    pares = df_cov_lower.stack().reset_index()
    pares.columns = ['Ativo_1', 'Ativo_2', 'Covariancia']
    
    # Filtramos APENAS os negativos e ordenamos do mais negativo para o menos negativo
    pares_negativos = pares[pares['Covariancia'] < 0].sort_values(by='Covariancia')
    
    caminho_pares = os.path.join(diretorio_dados, "pares_covariancia_negativa.csv")
    pares_negativos.to_csv(caminho_pares, index=False)
    
    print("\n=== RESUMO DO MAPEAMENTO ===")
    print(f"Total de cruzamentos possíveis: {int((len(df_cov)**2 - len(df_cov))/2)}")
    print(f"Pares com Covariância estritamente negativa encontrados: {len(pares_negativos)}")
    print(f"Heatmap salvo em: {caminho_grafico}")
    print(f"Tabela de Pares salva em: {caminho_pares}")
    print("====================================================")
    
    return pares_negativos

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

# Executa e guarda a lista de pares na variável
pares_hedge = mapear_covariancia_negativa(pasta_dados)

# Se quiseres ver os 10 pares com MAIOR covariância negativa na tela:
print("\nTop 10 Pares de Maior Proteção (Maior Covariância Negativa):")
print(pares_hedge.head(10).to_string(index=False))

=== INÍCIO DO MAPEAMENTO DE COVARIÂNCIA NEGATIVA ===
1. A carregar a matriz de covariância global...
2. A aplicar filtros matriciais (isolando covariâncias negativas)...
3. A desenhar o Mapa de Calor (Heatmap)...
4. A extrair os pares de ativos com proteção cruzada...

=== RESUMO DO MAPEAMENTO ===
Total de cruzamentos possíveis: 9180
Pares com Covariância estritamente negativa encontrados: 1
Heatmap salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\heatmap_covariancia_negativa.png
Tabela de Pares salva em: C:\VSCodeWorkspace\TCC_Escrito\data\pares_covariancia_negativa.csv

Top 10 Pares de Maior Proteção (Maior Covariância Negativa):
Ativo_1 Ativo_2  Covariancia
  RCSL4  GOLL54    -0.015884
